In [17]:
import torch
from torch import nn
from d2l import torch as d2l

# Source Token
# [B, T]
#    ↓ Encoder
# Encoder Outputs [T, B, H]
# Encoder State   [L, B, H]
#    ↓
# Fixed Context = Encoder Outputs[-1]
# [B, H]
#    ↓ 모든 Decoder Step에 복사
# [T, B, H]
#    ↓ Decoder Embedding과 결합
# [T, B, E + H]
#    ↓ Decoder GRU
# [T, B, H]
#    ↓ Linear
# [B, T, V]

In [18]:
# Sequence-To-Sequence Parameter Initialization

def init_seq2seq(
    module: nn.Module,
) -> None: 
    
    # Linear weights Initialization
    if type(module) is nn.Linear:
        nn.init.xavier_uniform_(
            module.weight
        )
        
    # GRU weights Initialization
    if type(module) is nn.GRU:
        for name, parameter in (
            module.named_parameters()
        ):
            if "weight" in name:
                nn.init.xavier_uniform_(
                    parameter
                )
    
    
    
# Sequence-To-Sequence Encoder

class Seq2SeqEncoder(d2l.Encoder):

    def __init__(
        self,
        vocab_size: int,
        embed_size: int,
        num_hiddens: int,
        num_layers: int,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_size,
        )

        self.rnn = d2l.GRU(
            num_inputs=embed_size,
            num_hiddens=num_hiddens,
            num_layers=num_layers,
            dropout=dropout,
        )

        self.apply(
            init_seq2seq
        )
        
    
    def forward(
        self,
        X: torch.Tensor,
        *args: object,
    ) -> tuple[
        torch.Tensor,
        torch.Tensor,
    ]:
        # Input Token: [B, T]
        
        
        # [nn.Embedding] 
        # embeddings: [T, B, E]
        embeddings = self.embedding(
            X.T.to(dtype=torch.long)
        )
        
        # [nn.GRU]
        # outputs: [T, B, H]
        # state  : [L, B, H]
        outputs, state = self.rnn(
            embeddings 
        )
        
        return (
            outputs,
            state,
        )

In [19]:
# Encoder Tensor Shape checking

vocab_size = 10
embed_size = 8
num_hiddens = 16
num_layers = 2

batch_size = 4
num_steps = 9

encoder = Seq2SeqEncoder(
    vocab_size=vocab_size,
    embed_size=embed_size,
    num_hiddens=num_hiddens,
    num_layers=num_layers,
)

X = torch.zeros(
    (
        batch_size,
        num_steps,
    ),
    dtype=torch.long,
)

encoder.eval()

with torch.no_grad():
    (
        encoder_outputs,
        encoder_state,
    ) = encoder(X)


# [B, T] = [4, 9]
print(
    "Input shape:",
    tuple(X.shape),
)

# 모든 Encoder Time Step의 Top Layer Output
# [T, B, H] = [9, 4, 16]
print(
    "Encoder Outputs shape:",
    tuple(encoder_outputs.shape),
)

# 모든 Encoder Layer의 Final State
# [L, B, H] = [2, 4, 16]
print(
    "Encoder State shape:",
    tuple(encoder_state.shape),
)

# 마지막 Time Step의 Top Layer Output
# [T, B, H] -> [B, H]
last_time_output = (
    encoder_outputs[-1]
)

# 마지막 Layer의 Final State
# [L, B, H] -> [B, H]
top_layer_final_state = (
    encoder_state[-1]
)

print(
    "\nLast Time Output shape:",
    tuple(last_time_output.shape),
)
print(
    "Top Layer Final State shape:",
    tuple(top_layer_final_state.shape),
)

d2l.check_shape(
    encoder_outputs,
    (
        num_steps,
        batch_size,
        num_hiddens,
    ),
)

d2l.check_shape(
    encoder_state,
    (
        num_layers,
        batch_size,
        num_hiddens,
    ),
)

# GRU의 Top Layer에서 마지막 Output과
# Final State는 동일하다.
torch.testing.assert_close(
    last_time_output,
    top_layer_final_state,
)

print(
    "\nLast Output equals Final State."
)

Input shape: (4, 9)
Encoder Outputs shape: (9, 4, 16)
Encoder State shape: (2, 4, 16)

Last Time Output shape: (4, 16)
Top Layer Final State shape: (4, 16)

Last Output equals Final State.


In [ ]:
# Sequence-To-Sequence Decoder

class Seq2SeqDecoder(d2l.Decoder):
    
    def __init__(
        self,
        vocab_size: int,
        embed_size: int,
        num_hiddens: int,
        num_layers: int,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_size,
        )
        
        # Decoder GRU Input: [E + H]
        self.rnn = d2l.GRU(
            num_inputs=(
                embed_size     # Embedding: [E]
                + num_hiddens  # Context  : [H]
            ),
            num_hiddens=num_hiddens,
            num_layers=num_layers,
            dropout=dropout,
        )
        
        # Transform Hidden state(H) to Vocabulary logit(V)
        self.dense = nn.LazyLinear(
            out_features=vocab_size
        )
        
        self.apply(
            init_seq2seq
        )
        


    def init_state(
        self,
        enc_all_outputs: tuple[
            torch.Tensor,
            torch.Tensor,
        ],
        *args: object,
    ) -> tuple[
        torch.Tensor,
        torch.Tensor,
    ]:
        
        # Encoder가 반환한 다음 두 Tensor를 
        # Decoder state로 그대로 전달
        #
        # encoder_outputs: [T, B, H]
        # encoder_state  : [L, B, H]
        return enc_all_outputs
    
    
    
    def forward(
        self,
        X: torch.Tensor,
        state: tuple[
            torch.Tensor,
            torch.Tensor,
        ],
    ) -> tuple[
        torch.Tensor,
        tuple[
            torch.Tensor,
            torch.Tensor,
        ],
    ]:
        # Decoder input X: [B, T]
    
    
        # [T, B, E]
        embeddings = self.embedding(
            X.T.to(dtype=torch.long)
        )

        (
            encoder_outputs, # [T, B, H]
            hidden_state,    # [L, B, H]
        ) = state


        # 모든 Encoder output 중 마지막 time step 선택: [B, H]
        fixed_context = (
            encoder_outputs[-1]
        )


        # fixed_context를 Decoder Time Step에 T개로 복사: [T, B, H]
        repeated_context = (
            fixed_context
            .unsqueeze(0)
            .repeat(
                embeddings.shape[0],
                1,
                1,
            )
        )
        
        # Feature dimension으로 Concatenate: [T, B, E + H]
        decoder_inputs = torch.cat(
            (
                embeddings,       # [T, B, E]
                repeated_context, # [T, B, H]
            ),
            dim=-1
        )
        
        # Decoder GRU
        # outputs      : [T, B, H]
        # hidden_state : [L, B, H]
        outputs, hidden_state = self.rnn(
            decoder_inputs, # [T, B, E + H]
            hidden_state,   # [L, B, H]
        )
        
        # Linear: H → V & 축 교환:
        # [T, B, H] -> [T, B, V] -> [B, T, V]
        logits = self.dense(
            outputs
        ).swapaxes(
            0,
            1,
        )
        
        return logits, (
            encoder_outputs,
            hidden_state,
        )

In [21]:
# Decoder Tensor Shape Review

decoder = Seq2SeqDecoder(
    vocab_size=vocab_size,
    embed_size=embed_size,
    num_hiddens=num_hiddens,
    num_layers=num_layers,
)

# Encoder Output과 State로 Decoder State 초기화
decoder_state = decoder.init_state((
    encoder_outputs,
    encoder_state,
))

decoder.eval()

with torch.no_grad():
    (
        decoder_outputs,
        updated_state,
    ) = decoder(
        X,
        decoder_state,
    )

print(
    "Decoder Input shape:",
    tuple(X.shape),
)
print(
    "Encoder Outputs shape:",
    tuple(encoder_outputs.shape),
)
print(
    "Encoder State shape:",
    tuple(encoder_state.shape),
)
print(
    "Decoder Outputs shape:",
    tuple(decoder_outputs.shape),
)
print(
    "Updated Hidden State shape:",
    tuple(updated_state[1].shape),
)

d2l.check_shape(
    decoder_outputs,
    (
        batch_size,
        num_steps,
        vocab_size,
    ),
)

d2l.check_shape(
    updated_state[1],
    (
        num_layers,
        batch_size,
        num_hiddens,
    ),
)

Decoder Input shape: (4, 9)
Encoder Outputs shape: (9, 4, 16)
Encoder State shape: (2, 4, 16)
Decoder Outputs shape: (4, 9, 10)
Updated Hidden State shape: (2, 4, 16)
